In [0]:
# Databricks notebook source

# MAGIC %run ./01-config

# COMMAND ----------

class HistoryLoader:

    def __init__(self, catalog="dev"):

        conf = Config()

        # Zone contenant les données du projet
        self.landing_zone = conf.base_dir_data + "/raw"

        # Données utilisées pour les chargements initiaux
        self.test_data_dir = conf.base_dir_data + "/test_data"

        # Unity Catalog
        self.catalog = catalog

        # Notre architecture Medallion
        self.silver_schema = conf.silver_schema


    # --------------------------------------------------
    # LOAD DATE LOOKUP
    # --------------------------------------------------

    def load_date_lookup(self):

        print("Loading date_lookup table...", end="")

        spark.sql(f"""
            INSERT OVERWRITE TABLE
            {self.catalog}.{self.silver_schema}.date_lookup

            SELECT
                date,
                week,
                year,
                month,
                dayofweek,
                dayofmonth,
                dayofyear,
                week_part

            FROM json.`{self.test_data_dir}/6-date-lookup.json/`
        """)

        print("Done")


    # --------------------------------------------------
    # LOAD ALL HISTORICAL / LOOKUP DATA
    # --------------------------------------------------

    def load_history(self):

        import time

        start = int(time.time())

        print("\nStarting historical data load...")

        self.load_date_lookup()

        print(
            f"Historical data load completed in "
            f"{int(time.time()) - start} seconds"
        )


    # --------------------------------------------------
    # GENERIC COUNT VALIDATION
    # --------------------------------------------------

    def assert_count(self, table_name, expected_count):

        print(
            f"Validating record counts in {table_name}...",
            end=""
        )

        actual_count = (
            spark.read
            .table(
                f"{self.catalog}.{self.silver_schema}.{table_name}"
            )
            .count()
        )

        assert actual_count == expected_count, (
            f"Expected {expected_count:,} records, "
            f"found {actual_count:,} in {table_name}"
        )

        print(
            f"Found {actual_count:,} / "
            f"Expected {expected_count:,} records: Success"
        )


    # --------------------------------------------------
    # VALIDATE HISTORY LOAD
    # --------------------------------------------------

    def validate(self):

        import time

        start = int(time.time())

        print("\nStarting historical data load validation...")

        self.assert_count(
            "date_lookup",
            365
        )

        print(
            f"Historical data load validation completed in "
            f"{int(time.time()) - start} seconds"
        )

### Description de 03-history-loader

Ce notebook implémente un **History Loader**, c’est-à-dire un composant chargé d’initialiser la plateforme avec des données historiques ou des données de référence avant le démarrage des traitements réguliers.

Dans ce projet, il est principalement utilisé pour charger la table `date_lookup`, une table calendrier utilisée dans la couche Silver.

Le fonctionnement est le suivant :

`ADLS Data Zone → HistoryLoader → dev.silver.date_lookup`

Le notebook contient plusieurs fonctions :

- `load_date_lookup()` : lit le fichier JSON contenant les données calendrier et les charge dans la table `date_lookup`.
- `load_history()` : lance l’ensemble des chargements historiques ou de référence.
- `assert_count()` : vérifie que le nombre de lignes chargées correspond au nombre attendu.
- `validate()` : exécute les contrôles permettant de vérifier que le chargement s’est correctement déroulé.

Le History Loader est particulièrement utile lorsqu’une entreprise possède déjà des données avant la mise en production du nouveau Lakehouse.

Par exemple :

`Données historiques existantes → chargement initial une seule fois → pipelines batch/streaming réguliers`

L’objectif est donc de **préparer et initialiser les données nécessaires au projet avant le fonctionnement normal des pipelines Bronze, Silver et Gold**.